<a href="https://colab.research.google.com/github/Zhalil24/BreastMRI-CNN-Classification/blob/main/VitNewResult.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================

import os
import shutil
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive


# ============================================================
# 2. DRIVE BAĞLAMA VE DATASET AÇMA
# ============================================================

drive.mount('/content/drive')

!cp "/content/drive/MyDrive/data_set_zip/breast_mri_dataset.rar" /content/
!unrar x -o+ /content/breast_mri_dataset.rar /content/


# ============================================================
# 3. AYARLAR VE CİHAZ
# ============================================================

DATA_DIR = "/content/breast_mri_dataset"
BATCH_SIZE = 32
NUM_EPOCHS = 10
IMG_SIZE = 224
NUM_CLASSES = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {device}")

# GPU belleği yetmezse BATCH_SIZE = 16 veya 8 yapılabilir.


# ============================================================
# 4. TRANSFORMASYONLAR
# ============================================================
# Burada data augmentation yok.
# Amaç: Fine-tuning edilmiş ViT-B/16 modelini CNN modelleriyle karşılaştırmak.

vit_mean = [0.485, 0.456, 0.406]
vit_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=vit_mean, std=vit_std)
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=vit_mean, std=vit_std)
])


# ============================================================
# 5. DATASETLER
# ============================================================

full_train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "test"),
    transform=test_transform
)

class_names = full_train_dataset.classes
print("Sınıflar:", full_train_dataset.class_to_idx)

if len(class_names) == 2:
    display_class_names = ["Benign", "Malignant"]
else:
    display_class_names = class_names


# ============================================================
# 6. TRAIN / VALIDATION SPLIT
# ============================================================

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# 7. ViT-B/16 MODEL OLUŞTURMA VE FINE-TUNING
# ============================================================

def build_vit_model():
    model = models.vit_b_16(
        weights=models.ViT_B_16_Weights.IMAGENET1K_V1
    )

    # İlk katmanlar donduruluyor.
    # Fine-tuning: Son encoder katmanları ve head eğitiliyor.
    for name, param in model.named_parameters():
        if (
            "encoder.layers.encoder_layer_0" in name or
            "encoder.layers.encoder_layer_1" in name or
            "encoder.layers.encoder_layer_2" in name or
            "encoder.layers.encoder_layer_3" in name or
            "encoder.layers.encoder_layer_4" in name or
            "conv_proj" in name
        ):
            param.requires_grad = False
        else:
            param.requires_grad = True

    model.heads.head = nn.Linear(
        model.heads.head.in_features,
        NUM_CLASSES
    )

    return model.to(device)


model = build_vit_model()

criterion = nn.CrossEntropyLoss()

# Eski yapı korunuyor: tek optimizer, tek learning rate.
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)


# ============================================================
# 8. EĞİTİM VE VALIDATION FONKSİYONLARI
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc


# ============================================================
# 9. MODEL EĞİTİMİ
# ============================================================

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

print("\n--- Fine-Tuning ViT-B/16 Eğitimi Başlıyor ---")

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS}: "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )


# ============================================================
# 10. TEST DEĞERLENDİRME
# ============================================================

model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)


# ============================================================
# 11. TEST METRİKLERİ
# ============================================================

test_accuracy = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, zero_division=0)
test_recall = recall_score(all_labels, all_preds, zero_division=0)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)
test_auc = roc_auc_score(all_labels, all_probs)

print("\n--- TEST SETİ PERFORMANS SONUÇLARI ---")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_auc:.4f}")


# ============================================================
# 12. ACCURACY GRAFİĞİ
# ============================================================

epochs_range = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(9, 6))
plt.plot(
    epochs_range,
    history["train_acc"],
    marker="o",
    label="Train Accuracy"
)
plt.plot(
    epochs_range,
    history["val_acc"],
    marker="o",
    label="Validation Accuracy"
)

plt.title("ViT-B/16 Fine-Tuning - Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 13. LOSS GRAFİĞİ
# ============================================================

plt.figure(figsize=(9, 6))
plt.plot(
    epochs_range,
    history["train_loss"],
    marker="o",
    label="Train Loss"
)
plt.plot(
    epochs_range,
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.title("ViT-B/16 Fine-Tuning - Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=display_class_names,
    yticklabels=display_class_names
)

plt.title("ViT-B/16 Fine-Tuning - Confusion Matrix")
plt.xlabel("Tahmin")
plt.ylabel("Gerçek")
plt.tight_layout()
plt.show()


# ============================================================
# 15. ROC CURVE
# ============================================================

fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr,
    tpr,
    lw=2,
    label=f"ROC Curve - AUC = {test_auc:.2f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    lw=2,
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ViT-B/16 Fine-Tuning - ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()